# Feature Attributions

In [ ]:
import sys
sys.path.append('../../../')
from polygene.eval.metrics import prepare_cell
import json
import torch
import requests
import scanpy as sc
from tqdm import tqdm
import torch.nn.functional as F
import pandas as pd, numpy as np
from polygene.data_utils.tokenization import GeneTokenizer, normalise_str
from scipy.stats import fisher_exact, ttest_ind
from sklearn.metrics import mutual_info_score

class AttributionAnalysis():
    def __init__(self, model, tokenizer: GeneTokenizer , data = None, device="cuda:0", biotype_json="../data_utils/vocab/gene_biotypes.json", ensembl_json="../data_utils/vocab/ensembl_to_gene.json"):
        """
        Initialize Analyzer that computes feature attributions with different methods for alpha group of cells
        model: PyTorch Model
        tokenizer: tokenizer saved from pickle
        data: str or AnnData h5ad file with cells of interest
        """
        self.model, self.tok = model, tokenizer
        self.device = device
        model.to(device)
        self.data = sc.read_h5ad(data) if isinstance(data, str) else data
        self.list_of_ensembl_ids = list(self.tok.gene_type_id_map.keys())

        self.ensembl_id_to_gene_name = json.load(open(ensembl_json)) if ensembl_json is not None else None
        self.gene_biotypes = json.load(open(biotype_json)) if biotype_json is not None else None
    
    def gradients(self, phenotype="disease", reduction_func = None, only_protein_encoding=True, disable_pbar=False, force_phenotype_value=None, use_data_slice=None):
        # reduction function needs to take alpha (S, D) matrix and return an (S,) vector. 
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        
        data = self.data if use_data_slice is None else use_data_slice
        cell_attributions = []
        for cell in tqdm(data, disable=disable_pbar):
            x = prepare_cell(cell, self.tok)
            x['input_ids'][torch.arange(1, 1+len(self.tok.phenotypic_types))] = self.tok.token_to_id_map[self.tok.mask_token]
            phenotype_value = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)] if force_phenotype_value is None else normalise_str(force_phenotype_value)
            x['inputs_embeds'] = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach().requires_grad_(True)

            y = F.softmax(self.model(**{k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k != 'str_labels'}).logits, dim=-1) # (B, S, D)
            Ly = y[0, 1+self.tok.phenotypic_types.index(phenotype), self.tok.flattened_tokens.index(phenotype_value)]
            self.model.zero_grad()
            Ly.backward()

            attributions_per_gene = reduction_func(x['inputs_embeds'].grad.detach().cpu().numpy()[self.tok.gene_token_type_offset:])
            gene_ensembl_ids = [self.list_of_ensembl_ids[gene - self.tok.gene_token_type_offset] for gene in x['token_type_ids'].detach().cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append( {k:v for k,v in zip(gene_ensembl_ids, attributions_per_gene)})

        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding: self.cell_attributions_df = self.select_protein_encoding_genes(self.cell_attributions_df)
        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df

    def integrated_gradients(self, phenotype="disease", reduction_func=None, steps=10, only_protein_encoding=True, disable_pbar=False):
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        cell_attributions = []
        for cell in tqdm(self.data, disable=disable_pbar):
            x = prepare_cell(cell, self.tok)
            x['input_ids'][torch.arange(1, 1+len(self.tok.phenotypic_types))] = self.tok.token_to_id_map[self.tok.mask_token]
            phenotype_value = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]

            target = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach()
            baseline = self.get_padded_baseline(x)
            gradients = torch.zeros_like(target)

            for alpha in torch.linspace(0, 1, steps):
                interpolated_input = (baseline + alpha*target).detach().requires_grad_(True)
                x['inputs_embeds']=interpolated_input.unsqueeze(0)

                y = F.softmax(self.model(**{k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k != 'str_labels'}).logits, dim=-1) # (B, S, D)
                Ly = y[0,1+self.tok.phenotypic_types.index(phenotype),self.tok.flattened_tokens.index(phenotype_value)]
                self.model.zero_grad()
                Ly.backward(retain_graph=True)
                gradients += interpolated_input.grad.detach()

            integrated_gradients = ( (target - baseline) * gradients / steps).detach().cpu().numpy()
            attributions_per_gene = reduction_func(integrated_gradients[self.tok.gene_token_type_offset:])
            
            gene_ensembl_ids = [self.list_of_ensembl_ids[i-self.tok.gene_token_type_offset] for i in x['token_type_ids'].cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append(dict(zip(gene_ensembl_ids,attributions_per_gene)))
        
        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding: self.cell_attributions_df = self.select_protein_encoding_genes(self.cell_attributions_df)
        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df
    
    def deep_lift(self, phenotype="disease", reduction_func=None, only_protein_encoding=True, disable_pbar=True):
        from captum.attr import DeepLift
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        cell_attributions=[]
        for cell in tqdm(self.data, disable=disable_pbar):
            x = prepare_cell(cell, self.tok); phenotype_value = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]
            feed = {k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k!='str_labels'}
            feed['inputs_embeds'] = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach().requires_grad_(True).unsqueeze(0)
            base = self.get_padded_baseline(x).unsqueeze(0)

            class ForwardWrapper(torch.nn.Module):
                def __init__(sself, model): super().__init__(); sself.model=model
                def forward(sself, inputs_embeds, attention_mask): return sself.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits
            dl = DeepLift(ForwardWrapper(self.model))

            target = (1+self.tok.phenotypic_types.index(phenotype),self.tok.flattened_tokens.index(phenotype_value))

            DeepLIFT_attributions = dl.attribute(inputs=feed['inputs_embeds'], baselines=base, additional_forward_args=(feed["attention_mask"],), target=target)
            attributions_per_gene = reduction_func(DeepLIFT_attributions[0].detach().cpu().numpy()[self.tok.gene_token_type_offset:])
            gene_ensembl_ids = [self.list_of_ensembl_ids[i-self.tok.gene_token_type_offset] for i in x['token_type_ids'].cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append(dict(zip(gene_ensembl_ids, attributions_per_gene)))
        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding: self.cell_attributions_df = self.select_protein_encoding_genes(self.cell_attributions_df)
        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df
    
    def disease_vector(self, phenotype="disease", reduction_func=None, only_protein_encoding=True, disable_pbar=True):
        from captum.attr import DeepLift
        reduction_func = (lambda m: np.linalg.norm(m, axis=1)) if reduction_func is None else reduction_func
        cell_attributions=[]
        for cell in tqdm(self.data, disable=disable_pbar):
            x = prepare_cell(cell, self.tok); phenotype_value = x['str_labels'][1+self.tok.phenotypic_types.index(phenotype)]
            feed = {k:v.unsqueeze(0).to(self.device) for k,v in x.items() if k!='str_labels'}
            feed['inputs_embeds'] = self.model.embeddings(x['input_ids'].to(self.device), x['token_type_ids'].to(self.device)).detach().requires_grad_(True).unsqueeze(0)
            base = self.get_padded_baseline(x).unsqueeze(0)

            class ForwardWrapper(torch.nn.Module):
                def __init__(sself, model): super().__init__(); sself.model=model
                def forward(sself, inputs_embeds, attention_mask): return sself.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits
            dl = DeepLift(ForwardWrapper(self.model))

            target = (1+self.tok.phenotypic_types.index(phenotype),self.tok.flattened_tokens.index(phenotype_value))

            DeepLIFT_attributions = dl.attribute(inputs=feed['inputs_embeds'], baselines=base, additional_forward_args=(feed["attention_mask"],), target=target)
            attributions_per_gene = reduction_func(DeepLIFT_attributions[0].detach().cpu().numpy()[self.tok.gene_token_type_offset:])
            gene_ensembl_ids = [self.list_of_ensembl_ids[i-self.tok.gene_token_type_offset] for i in x['token_type_ids'].cpu().numpy()[self.tok.gene_token_type_offset:]]
            cell_attributions.append(dict(zip(gene_ensembl_ids, attributions_per_gene)))
        self.cell_attributions_df = pd.DataFrame(cell_attributions).fillna(0)
        if only_protein_encoding: self.cell_attributions_df = self.select_protein_encoding_genes(self.cell_attributions_df)
        self.cell_attributions_df.columns = pd.Series(self.cell_attributions_df.columns).apply(lambda ensembl: self.ensembl_id_to_gene_name[ensembl])
        return self.cell_attributions_df
    
    def get_padded_baseline(self, x):
        x_baseline = {
            "input_ids": torch.cat([x['input_ids'][:self.tok.gene_token_type_offset],
                                     torch.tensor([self.tok.convert_tokens_to_ids(self.tok.pad_token)] * len(x['input_ids'][self.tok.gene_token_type_offset:]),
                                                   device=x['input_ids'].device, dtype=x['input_ids'].dtype)] ),
            "token_type_ids": torch.cat([x['token_type_ids'][:self.tok.gene_token_type_offset],
                                     torch.tensor([0] * len(x['input_ids'][self.tok.gene_token_type_offset:]), 
                                                  device=x['input_ids'].device, dtype=x['token_type_ids'].dtype)], ),
        }
        baseline_input_embeds = self.model.embeddings(x_baseline['input_ids'].to(self.device), x_baseline['token_type_ids'].to(self.device)).detach()
        return baseline_input_embeds
    
    def select_protein_encoding_genes(self, attributions_df):
        ensembl_ids =  attributions_df.columns.tolist()
        mask = [eid for eid in ensembl_ids if self.gene_biotypes.get(eid, "") == "protein_coding"]
        return attributions_df.loc[:, np.array(mask)]

    @staticmethod
    def get_associated_genes(disease_name, disease_ontology_term_id, top=100):
        url = "https://api.platform.opentargets.org/api/v4/graphql"
        disease_ontology_term_id = disease_ontology_term_id.replace(':', '_')

        query_assoc = """
            query associatedTargets($diseaseId: String!, $size: Int!) {
                disease(efoId: $diseaseId) {
                    id
                    name
                    associatedTargets(page: { index: 0, size: $size }) {
                        count
                        rows { target { id approvedSymbol } score }
                    }
                }
            }
        """

        variables = {"diseaseId": disease_ontology_term_id, "size": top}
        resp = requests.post(url, json={"query": query_assoc, "variables": variables})
        try:
            response_json = resp.json()
            response = response_json.get('data', {}).get('disease')
        except ValueError:
            return {}
        
        if response is None:
            query_search = """
                query search($term: String!) {
                    search(queryString: $term, entityNames: ["disease"]) {
                        hits { id name }
                    }
                }
            """
            #search_resp = requests.post(url, json={"query": query_search, "variables": {"term": disease_name}}).json()
            resp = requests.post(url, json={"query": query_search, "variables": {"term": disease_name}})
            try:
                search_resp = resp.json()
            except ValueError:
                return {}
            hits = search_resp.get('data', {}).get('search', {}).get('hits', [])
            if hits:
                new_id = hits[0]['id']
                variables = {"diseaseId": new_id, "size": top}
                response = requests.post(url, json={"query": query_assoc, "variables": variables}).json()['data']['disease']
            else:
                return {}

        if response:
            return {row['target']['approvedSymbol']: row['score'] for row in response['associatedTargets']['rows'][:top]}
        return {}
    
    def validate_attributions(self, k=100, phenotype_obs_key="disease", phenotype_obs_value="normal", baseline=None, overwrite_ontology_id=None):
        ontology_id = self.data.obs[self.data.obs[phenotype_obs_key] == phenotype_obs_value]['disease_ontology_term_id'].tolist()[0] if overwrite_ontology_id is None else overwrite_ontology_id
        attributed_genes = self.cell_attributions_df.loc[(self.data.obs[phenotype_obs_key] == phenotype_obs_value).tolist(),:].sum(axis=0) if baseline is None else baseline
        
        opentarget_dict = self.get_associated_genes(phenotype_obs_key, ontology_id, k)
        opentarget_genes = set(opentarget_dict.keys())
        topk_attributed = set(attributed_genes.sort_values(ascending=False)[:k].index.tolist())

        overlap_genes = opentarget_genes & topk_attributed
        overlap = len(overlap_genes)
        TP, FP, FN = overlap, k - overlap, k - overlap
        TN = len(attributed_genes) - (TP + FP + FN)
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        _, pval = fisher_exact([[TP,FP],[FN,TN]], alternative="greater") if all(x > 0 for x in  [TP, FP, FN, TN]) else (0, 1)
        candidate_genes = attributed_genes.sort_values(ascending=False)[:k].drop(list(overlap_genes)).index.tolist()

        max_overlap = opentarget_genes & set(attributed_genes.index.tolist())

        return {"recall": recall,
                "max_recall": overlap/max(len(max_overlap), 1),
                "fisher_pvalue": pval,
                "opentarget_genes": pd.Series(opentarget_dict),
                "attributed_genes":  attributed_genes,
                "overlap_candidate_genes": (sorted(overlap_genes, key=lambda x: opentarget_dict[x]), candidate_genes)
            }

    def baselines(self, phenotype_obs_key='disease', target_label=None, baseline_label=None, k=100):
        X = self.data.X.toarray()
        nonzero_mask = X.sum(axis=0) > 0
        X = X[:, nonzero_mask]
        var_names = self.data.var_names[nonzero_mask]

        y = self.data.obs[phenotype_obs_key].to_numpy()
        mask_target, mask_baseline = (y == target_label), (y == baseline_label)
        Xc, Xn = X[mask_target], X[mask_baseline]
        t_stat, _ = ttest_ind(Xc, Xn, axis=0, equal_var=False, nan_policy='omit')
        attributions_gwas = pd.Series(t_stat, index=[self.ensembl_id_to_gene_name.get(k,k) for k in var_names])
        gwas_eval = self.validate_attributions(phenotype_obs_key=phenotype_obs_key, phenotype_obs_value=target_label, baseline=attributions_gwas, k=k)

        y_bin = np.where(y == target_label, 1, 0)
        mi_scores = []
        for j in range(X.shape[1]):
            xj = np.asarray(X[:, j]).ravel()
            bins = np.quantile(xj, np.linspace(0, 1, 6))
            xj_disc = np.digitize(xj, bins, right=True)
            mi_scores.append(mutual_info_score(xj_disc, y_bin))

        attributions_mutual_information = pd.Series(mi_scores, index=[self.ensembl_id_to_gene_name.get(k,k) for k in var_names])
        mutual_information_eval = self.validate_attributions(phenotype_obs_key=phenotype_obs_key, phenotype_obs_value=target_label, baseline=attributions_mutual_information, k=k)
        return gwas_eval, mutual_information_eval ,attributions_gwas, attributions_mutual_information


/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## recall @ g, wasserstein of recall distributions, 

In [ ]:
import json,os
import sys
sys.path.append('../../../')
import scanpy as sc
import pandas as pd, numpy as np
from polygene.analysis.attributions.attributions import AttributionAnalysis
from polygene.model.model import load_trained_model
from tqdm import tqdm
from polygene.analysis.geodesics.geodesics import Geodesic
DIR =  "/media/lleger/LaCie/mit/disease_geometry/attributions/"
random_state=3
K = 1000
n_cells = 100

files = os.listdir(DIR)
model, tokenizer = load_trained_model("../../../saved_models/gesam_polygene_run_4/", checkpoint_n=-1)
tokenizer.bypass_inference=True
decoder = model.prediction_head
analyzer = AttributionAnalysis(model, tokenizer, biotype_json="../../data_utils/vocab/gene_biotypes.json", ensembl_json="../../data_utils/vocab/ensembl_to_gene.json")

results = {}
for file in files:
    if "cells" not in file: continue
    disease = file.split('_cells')[0]
    results[disease] = {}

    cells = sc.read_h5ad(DIR + file)
    manifold = pd.read_pickle(DIR + disease + "_embeddings.pkl")[0]
    cells.obs_names_make_unique()
    sampled = cells.obs.groupby("disease", group_keys=False).apply(lambda x: x.sample(n=min(n_cells, len(x)), random_state=random_state)).index
    analyzer.data = cells[sampled]
    gwas_eval, mutual_information_eval = analyzer.baselines(k=K, baseline_label="normal", target_label=disease)

    # Disease Vector/ Information Geometry Attributions:
    normal_states = manifold[cells.obs_names.get_indexer(sampled)][analyzer.data.obs['disease'] == "normal"]
    disease_states = manifold[cells.obs_names.get_indexer(sampled)][analyzer.data.obs['disease'] == disease]
    disease_ontology_id = analyzer.data[analyzer.data.obs['disease'] == disease].obs['disease_ontology_term_id'].tolist()[0]
    disease_vector_attributions, geodesic_attributions = [], []
    for z0, z1 in zip(tqdm(normal_states), disease_states):
        geodesic = Geodesic(z0, z1, total_t=10, decoder=decoder, device=model.device)
        mask = geodesic.discretize(manifold=manifold, k=1)
        disease_vector_attributions.append( geodesic.path_integrated_gradients(
            cells, mask, model, tokenizer, phenotype_value=disease, disease_ontology_id=disease_ontology_id)['cumulative_integrated_gradients'] )
        geodesic.optimize(steps=500, lr=1e-3)
        mask = geodesic.discretize(manifold=manifold, k=1)
        geodesic_attributions.append( geodesic.path_integrated_gradients(
            cells, mask, model, tokenizer, phenotype_value=disease, disease_ontology_id=disease_ontology_id)['cumulative_integrated_gradients'] )

    disease_vector_attributions = pd.concat(disease_vector_attributions, axis=1).T.fillna(0).sum(axis=0)
    results[disease]['Disease Vector'] = analyzer.validate_attributions(k=K, phenotype_obs_value=disease,
                                                        overwrite_ontology_id=disease_ontology_id, baseline=disease_vector_attributions)

    geodesic_attributions =  pd.concat(geodesic_attributions, axis=1).T.fillna(0).sum(axis=0)
    results[disease]['Information Geometry'] = analyzer.validate_attributions(k=K, phenotype_obs_value=disease,
                                                        overwrite_ontology_id=disease_ontology_id, baseline=geodesic_attributions)
        
    results[disease]['DGE'] = gwas_eval
    results[disease]['Mutual Information'] = mutual_information_eval

    analyzer.data = analyzer.data[analyzer.data.obs['disease'] == disease]

    analyzer.gradients(force_phenotype_value=disease, disable_pbar=False)
    results[disease]['Gradients'] = analyzer.validate_attributions(k=K, phenotype_obs_value=disease, overwrite_ontology_id=disease_ontology_id)
    
    analyzer.integrated_gradients(force_phenotype_value=disease, disable_pbar=False)
    results[disease]['Integrated Gradients'] = analyzer.validate_attributions(k=K, phenotype_obs_value=disease, overwrite_ontology_id=disease_ontology_id)
    
    analyzer.deep_lift(force_phenotype_value=disease, disable_pbar=False)
    results[disease]['DeepLIFT'] = analyzer.validate_attributions(k=K, phenotype_obs_value=disease, overwrite_ontology_id=disease_ontology_id)


/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  1%|          | 1/100 [00:04<08:04,  4.89s/it]

In [2]:
disease

'Alzheimer Disease'

In [8]:
TABLE = pd.DataFrame(results)
TABLE
TABLE.applymap(lambda x: (x['recall'], x['fisher_pvalue']) if isinstance(x, dict) else None)

,small cell lung carcinoma,lung large cell carcinoma,glioblastoma,Alzheimer Disease
Disease Vector,"(0.065, 0.9999610576288853)","(0.071, 0.998377157856252)","(0.116, 0.0036466752070831495)",None
Information Geometry,"(0.063, 0.9999999064708379)","(0.065, 0.9999888064594965)","(0.098, 0.2577365781872063)",None
DGE,"(0.054, 1.0)","(0.06, 1.0)","(0.066, 1.0)",None
Mutual Information,"(0.068, 1.0)","(0.076, 1.0)","(0.075, 1.0)",None
Gradients,"(0.067, 1.0)","(0.062, 1.0)","(0.093, 1.0)",None
Integrated Gradients,"(0.051, 1.0)","(0.065, 1.0)","(0.083, 1.0)",None
DeepLIFT,"(0.076, 1.0)","(0.059, 1.0)","(0.084, 1.0)",None


In [ ]:

from polygene.model.model import load_trained_model
import scanpy as sc
SAVE_DIR = "/media/lleger/LaCie/mit/disease_vector/"
#model_path = "../../../runs/gesam_polygene_run_4/" #'/media/lleger/LaCie/POLYGENE/'
model, tokenizer = load_trained_model("../../../saved/gesam_polygene_run_4/", checkpoint_n=-1)

cells = sc.read_h5ad(args.data_path)
idx = cells.obs.groupby('disease', group_keys=False).apply(lambda x: x.sample(n=min(args.n_cells, len(x)))).index
cells = cells[idx[cells.obs.loc[idx, 'disease'].isin([args.disease_name, 'normal'])]]
analyzer = AttributionAnalysis(model, tokenizer, data=cells)

# BASELINE
r = analyzer.baselines(phenotype_obs_key="disease", case_label="Alzheimer disease", control_label="normal", method_top_attr="Q3", k=50)

cells = cells[cells.obs.loc[idx, 'disease'].isin([args.disease_name])]
analyzer.cells = cells

# Gradients
df = analyzer.gradients(only_protein_encoding=True)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease")
# Integrated Gradients
df = analyzer.integrated_gradients(only_protein_encoding=True)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease")
# DeepLIFT Gradients
df = analyzer.deep_lift(only_protein_encoding=True)
r = analyzer.validate_attributions(k=50, method_top_attr="Q3", phenotype_obs_value="Alzheimer disease")

usage: attributions program [-h] [--model_path MODEL_PATH]
                            [--data_path DATA_PATH] [--k K]
                            [--n_cells N_CELLS]
                            [--method_top_attr METHOD_TOP_ATTR]
                            [--disease_name DISEASE_NAME] [--baselines]
attributions program: error: unrecognized arguments: --f=/run/user/1012/jupyter/runtime/kernel-v3ed4d652e95f942158b88468d8403fea0ff7d503d.json


SystemExit: 2

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
